In [ ]:
import pandas as pd
import pickle
import json
import urllib.request
import numpy as np
import plotly.graph_objects as go
from typing import Sequence, Optional, Union, List, Tuple
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import xarray as xr

In [ ]:
path_DALEC = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/DALEC/ProcDalec_L2_202606_M99.pkl'
path_all_pic  = '/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/PIC_discrete/all_PIC.csv'



In [ ]:
# searching for station

df = pd.read_csv(path_all_pic)
df = df.sort_values('datetime')
groups = {k: v.copy() for k, v in df.groupby("Campaign")}
SS_campaigns= {k: v for k, v in groups.items() if k.startswith("SS")}

print('available Campaigns')
print(groups.keys())
print('available Simon Stevins Campaigns')
print(SS_campaigns.keys())

df = SS_campaigns['SS20250602']
df = df.where(df['Station'].str.endswith(('05', '06', '07', '09', '10', '11')))
df = df.dropna(axis = 'rows')
df = df.reset_index()


lons = list(df["lon"])
lats = list(df["lat"])
stations = list(df["Station"])
datetimes = list(df["datetime"])


In [ ]:
x = pickle.load(open(path_DALEC, 'rb'))
rrs = x['Rrs']
lat = x['Ancillary']['LATITUDE']
lon = x['Ancillary']['LONGITUDE'] 

print(lat)

In [ ]:
rrs.index = pd.to_datetime(rrs.index)
lat.index = pd.to_datetime(lat.index)
lon.index = pd.to_datetime(lon.index)

In [ ]:
dalec = pd.concat([rrs, lat, lon], axis=1, join="inner")
print(dalec.columns)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Convert index once
dalec = dalec.copy()
dalec.index = pd.to_datetime(dalec.index, utc=True)

# Ensure datetimes are Timestamps with the same timezone
datetimes = pd.to_datetime(datetimes, utc=True)

# Identify wavelength columns once
cols = dalec.columns
wavelen_cols = [
    c for c in cols
    if c not in ('LATITUDE', 'LONGITUDE') and np.isreal(c)
]

for dt in datetimes:
    # Find nearest index in dalec to this dt
    nearest_idx = dalec.index.reindex([dt], method='nearest')[0]

    row = dalec.loc[nearest_idx]

    plt.figure()
    plt.plot(wavelen_cols, row[wavelen_cols].values, marker='')
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Rrs (DALEC)')
    plt.title(f'Rrs spectrum at {dt} (nearest: {nearest_idx})')
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_rrs_spectra(
    ds: xr.Dataset,
    lons: Sequence[float],
    lats: Sequence[float],
    stations: Sequence[str],
    rrs_variable: str = "Rrs",
    wavelength_variable: str = "wavelength",
    mode: str = "multi",  # "single" or "multi"
    figsize: Tuple[float, float] = (6.5, 4.5),
    dpi: int = 150,
    xlim: Optional[Tuple[float, float]] = (400, 700),
    ylim: Optional[Tuple[float, float]] = None,
    xticks: Optional[Sequence[float]] = None,
    xlabel: str = "Wavelength (nm)",
    ylabel: str = r"Remote-sensing reflectance, $R_{rs}$ (sr$^{-1}$)",
    title: Optional[str] = None,
    legend_title: str = "Station",
    cmap: str = "viridis",
    cmap_range: Tuple[float, float] = (0, 0.9),
    line_kwargs: Optional[dict] = None,
    grid: bool = True,
    grid_kwargs: Optional[dict] = None,
    minor_ticks: bool = True,
    spines_top_right: bool = True,
    save_dir: Optional[str] = None,
    save_kwargs: Optional[dict] = None,
    show: bool = True,
    apply_style: bool = True,
) -> plt.Figure:
    """
    Plot Rrs spectra for one or multiple stations.

    Parameters
    ----------
    ds : xr.Dataset
        xarray Dataset containing the Rrs variable and wavelength coordinate.
    lons : sequence of float
        Longitudes for each station.
    lats : sequence of float
        Latitudes for each station.
    stations : sequence of str
        Station names/labels for each location.
    rrs_variable : str, optional
        Name of the Rrs variable in the Dataset.
    wavelength_variable : str, optional
        Name of the wavelength coordinate/variable.
    mode : {"single", "multi"}, optional
        "single" → create one figure per station (calls plt.show() each time).
        "multi" → plot all stations on a single figure.
    figsize : tuple, optional
        Figure size (width, height) in inches.
    dpi : int, optional
        Figure resolution.
    xlim : tuple or None, optional
        (xmin, xmax) for x-axis. Set to None to auto-scale.
    ylim : tuple or None, optional
        (ymin, ymax) for y-axis. Set to None to auto-scale.
    xticks : sequence or None, optional
        Tick locations on x-axis.
    xlabel : str, optional
        X-axis label.
    ylabel : str, optional
        Y-axis label.
    title : str or None, optional
        Figure title. If None, no title is added.
    legend_title : str, optional
        Title for the legend.
    cmap : str, optional
        Matplotlib colormap for station colors.
    cmap_range : tuple, optional
        Fractional range of colormap to use, e.g. (0, 0.9).
    line_kwargs : dict or None, optional
        Additional keyword arguments passed to `ax.plot()` for each spectrum.
        Example: {"linewidth": 0.6, "alpha": 0.85}.
    grid : bool, optional
        Whether to draw a grid.
    grid_kwargs : dict or None, optional
        Keyword arguments for `ax.grid()`, e.g. {"ls": ":", "alpha": 0.9}.
    minor_ticks : bool, optional
        Whether to add minor ticks on both axes.
    spines_top_right : bool, optional
        Whether to show top and right spines.
    save_path : str or None, optional
        Path (including filename) to save the figure. If None, figure is not saved.
    save_kwargs : dict or None, optional
        Additional keyword arguments for `fig.savefig()`, e.g. {"dpi": 300}.
    show : bool, optional
        Whether to call `plt.show()` at the end.
    apply_style : bool, optional
        Whether to apply a publication-style `rcParams` update.

    Returns
    -------
    fig : matplotlib.figure.Figure
        The created figure object.
    """

    if len(lons) != len(lats) or len(lons) != len(stations):
        raise ValueError("lons, lats, and stations must have the same length.")

    if line_kwargs is None:
        line_kwargs = {}

    if grid_kwargs is None:
        grid_kwargs = {"ls": ":", "alpha": 0.9}

    if save_kwargs is None:
        save_kwargs = {}

    if apply_style:
        plt.style.use("default")
        plt.rcParams.update(
            {
                "font.size": 11,
                "font.family": "sans-serif",
                "font.sans-serif": [
                    "Arial",
                    "DejaVu Sans",
                    "Liberation Sans",
                ],
                "axes.linewidth": 1.2,
                "axes.labelsize": 12,
                "axes.titlelocation": "left",
                "axes.titleweight": "normal",
                "xtick.major.size": 6,
                "xtick.major.width": 1.2,
                "xtick.minor.size": 3,
                "xtick.minor.width": 1.0,
                "ytick.major.size": 6,
                "ytick.major.width": 1.2,
                "ytick.minor.size": 3,
                "ytick.minor.width": 1.0,
                "xtick.direction": "in",
                "ytick.direction": "in",
                "xtick.top": True,
                "ytick.right": True,
                "legend.frameon": False,
                "legend.fontsize": 10,
            }
        )

    if mode == "single":
        # One figure per station
        for lon, lat, station in zip(lons, lats, stations):
            test = ds[rrs_variable].sel(
                longitude=lon,
                latitude=lat,
                method="nearest",
            )

            fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

            ax.plot(
                test[wavelength_variable].values,
                test.values,
                **line_kwargs,
            )

            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)

            if xlim is not None:
                ax.set_xlim(xlim)
            if xticks is not None:
                ax.set_xticks(xticks)
            if ylim is not None:
                ax.set_ylim(ylim)

            if title is not None:
                ax.set_title(title)
            else:
                ax.set_title(f"Station: {station}")

            if grid:
                ax.grid(**grid_kwargs)

            if minor_ticks:
                ax.xaxis.set_minor_locator(AutoMinorLocator())
                ax.yaxis.set_minor_locator(AutoMinorLocator())

            if spines_top_right:
                ax.spines["top"].set_visible(True)
                ax.spines["right"].set_visible(True)

            plt.tight_layout()

            if save_dir is not None:
                # For single mode, you might want to include station in filename
                fig.savefig(f'{save_dir}/{station}_rrs_spectrum.png', dpi=save_kwargs.get("dpi", dpi), bbox_inches="tight")

            if show:
                plt.show()
            else:
                plt.close(fig)

        # Return last figure (or None if you prefer)
        return fig

    elif mode == "multi":
        # All spectra in one plot
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

        colors = plt.cm.get_cmap(cmap)(np.linspace(cmap_range[0], cmap_range[1], len(stations)))

        for i, (lon, lat, station) in enumerate(zip(lons, lats, stations)):
            test = ds[rrs_variable].sel(
                longitude=lon,
                latitude=lat,
                method="nearest",
            )

            ax.plot(
                test[wavelength_variable].values,
                test.values,
                color=colors[i],
                label=f"{station}",
                **line_kwargs,
            )

        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)

        if xlim is not None:
            ax.set_xlim(xlim)
        if xticks is not None:
            ax.set_xticks(xticks)
        if ylim is not None:
            ax.set_ylim(ylim)

        if title is not None:
            ax.set_title(title)

        if grid:
            ax.grid(**grid_kwargs)

        if minor_ticks:
            ax.xaxis.set_minor_locator(AutoMinorLocator())
            ax.yaxis.set_minor_locator(AutoMinorLocator())

        if spines_top_right:
            ax.spines["top"].set_visible(True)
            ax.spines["right"].set_visible(True)

        if len(stations) > 0:
            ax.legend(title=legend_title, loc="upper right", frameon=False)

        plt.tight_layout()

        if save_dir is not None:
            fig.savefig(f"{save_dir}{stations[0][0:10]}_rrs_spectra_all.png", dpi=save_kwargs.get("dpi", dpi), bbox_inches="tight")

        if show:
            plt.show()

        return fig

    else:
        raise ValueError("mode must be 'single' or 'multi'.")